# 🚀 06 — Deploy SETU-Rail as a Databricks App

## Pre-requisites
- All pipeline notebooks (01–05) have been run successfully
- AI/BI dashboard has been published (from `05_dashboard/01_create_dashboard`)
- Vector Search index is ONLINE (`03_vani_rag_pipeline/02_build_vector_index`)
- MLflow model is registered with `@production` alias (`02_drishti_ml_pipeline/02_train_gbt_delay_model`)

## Step 1 — Enable dashboard embedding
1. Click your **username** (top right) → **Settings** → **Security**
2. Scroll to **Embed dashboards** → click **Allow**

## Step 2 — Get the dashboard embed URL
1. Open your SETU-Rail Dashboard
2. Click **Share** → **Embed**
3. Copy the URL (format: `https://<workspace>.cloud.databricks.com/embed/dashboardsv3/<id>`)
4. Open `06_application/app.yaml` and paste it next to `value:`

## Step 3 — Deploy the app
1. Left sidebar → **Compute** → **Apps** → **Create app**
2. Choose **Custom** → name it **`setu-rail-app`** → **Create app**
3. Wait for app to be created (~30 sec)
4. Click **Deploy** → **Deploy using different source code**
5. Source code path: point to the `06_application` folder in your repo
6. Click **Deploy** → wait for status **Running**
7. Click **Open app** → you'll see the 5-tab SETU-Rail interface

## Step 4 — Verify each tab
| Tab | Test | Expected |
|-----|------|----------|
| 🚂 Dhara | Enter train `12951`, station `NZM`, click Predict | Shows delay + SHAP bars |
| 💬 Vani | Ask "Am I entitled to compensation for 4hr delay?" | Answer with section citations |
| 🕸️ Cascade | Source `12951`, station `NZM`, delay 60 min | Table of affected trains |
| 🎫 Drishti | Click "Generate action card" | Action card with rights |
| 📊 Analytics | Wait 5 sec | Charts from live Delta queries |

## Troubleshooting
| Error | Fix |
|-------|-----|
| `DASHBOARD_URL not set` | Edit `app.yaml`, redeploy |
| Vani returns generic error | Check Vector Search endpoint status: `03_vani/02_build_vector_index` |
| Model load fails | Re-run `02_drishti/02_train_gbt_delay_model`, confirm `@production` alias |
| App stuck in Starting | Check Logs from app page for Python import errors |

In [0]:
# Verify all prerequisites are met before deploying
import mlflow
from databricks.vector_search.client import VectorSearchClient

checks = {}

# 1. Gold feature table
try:
    n = spark.table("setu_rail.gold.features_delay_ml").count()
    checks["gold.features_delay_ml"] = f"✅ {n:,} rows"
except Exception as e:
    checks["gold.features_delay_ml"] = f"❌ {e}"

# 2. Trained model
try:
    mlflow.set_registry_uri("databricks-uc")
    from mlflow import MlflowClient
    client = MlflowClient()
    v = client.get_model_version_by_alias("setu_rail.gold.setu_delay_predictor", "production")
    checks["MLflow model @production"] = f"✅ version {v.version}"
except Exception as e:
    checks["MLflow model @production"] = f"❌ {e}"

# 3. Vector Search
try:
    vs = VectorSearchClient()
    idx = vs.get_index(endpoint_name="setu_rail_vs", index_name="setu_rail.gold.rules_vs_index")
    desc = idx.describe()
    status = desc.get("status", {}).get("detailed_state", "UNKNOWN")
    checks["Vector Search index"] = f"✅ {status}" if "ONLINE" in status else f"⚠️ {status}"
except Exception as e:
    checks["Vector Search index"] = f"❌ {e}"

# 4. Rules chunks
try:
    n = spark.table("setu_rail.silver.rules_chunks").count()
    checks["silver.rules_chunks"] = f"✅ {n:,} chunks"
except Exception as e:
    checks["silver.rules_chunks"] = f"❌ {e}"

print("\n🔍 Pre-deployment checklist:")
for k, v in checks.items():
    print(f"  {k:<40} {v}")